# Simple bridge, pre-trained on public English → Hinglish data, then fine-tuned on the personas

```
"<voice>: <English sentence>"  →  ModernBERT-base + LoRA  →  bridge (768 → 2560)                    [GPU 0]
                                                                  │ memory
<bos>  →  Gemma 3 4B IT (4-bit) + LoRA, cross-attention after every 4th layer  →  Hinglish  <end_of_turn>   [GPU 1]
```

The architecture is the one in `7-simple-bridge/` (the decoder gets nothing but the bridge; two LoRAs + bridge + cross-attention trained). That run showed the bridge cannot learn translation from 768 persona pairs: every output collapsed to one stock line per persona. So the bridge first learns **plain translation** from a public corpus, then the personas.

| Stage | Data | Encoder input | Pairs | What it teaches |
|---|---|---|---|---|
| **1. Translation pre-training** | Hinglish-TOP + PHINC (below) | `Plain: <English>` | ≈ 24,000, one pass | carrying an English sentence's meaning through the bridge into Hinglish |
| **2. Persona fine-tuning** | v4 merged | `Don: <English>` (etc.) | 768, up to 6 epochs | the three voices, continuing from stage 1 |

**Public corpora** (licences checked 2026-10-19; revisions pinned in Cell 1):

| Corpus | Licence | Used |
|---|---|---|
| **Hinglish-TOP** (Google Research; Hugging Face mirror `rvv-karma/English-Hinglish-TOP`) — English assistant queries with human (6.5k) and synthetic (170k) Hinglish | Apache-2.0 | all human pairs, synthetic pairs to fill up to 24,000 |
| **PHINC** (Srivastava & Singh, WNUT 2020; `veezbo/phinc`) — 13,738 Hinglish tweets with human English translations | CC BY 4.0 (attribution: the PHINC paper) | cleaned (no handles, links, hashtags; sensible lengths), up to 12,000 |

**Hygiene:** any corpus pair whose English resembles a v4 validation, test, out-of-domain or minimal-pair sentence (word overlap ≥ 0.6) is dropped; 400 corpus pairs (200 from each) are held out from stage 1 to measure it.

**Reported:** stage-1 held-out loss during training, chrF and examples on held-out corpus pairs after stage 1; then, after stage 2, chrF, stopping, distinct outputs and examples on v4 validation, test and out-of-domain — the same measures as `7-simple-bridge/` (validation chrF 12.0, 5 distinct outputs of 75).

In [2]:
# Cell 0 — Hugging Face token (Gemma 3 is gated). Clear it before saving or sharing the notebook.

HF_TOKEN = ""

## Cell 1 — Setup, pinned revisions

Two GPUs (Kaggle: *GPU T4 × 2*). Models and both corpora are pinned to the revision found now.

In [3]:
# Cell 1 — Packages, settings, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2" -U bitsandbytes

import json, math, os, random, re, statistics, time, urllib.request
from collections import Counter

import pandas as pd
import peft
import torch
import torch.nn as nn
import transformers
from huggingface_hub import HfApi, login

ENCODER, DECODER = "answerdotai/ModernBERT-base", "google/gemma-3-4b-it"
TOP_REPO, PHINC_REPO = "rvv-karma/English-Hinglish-TOP", "veezbo/phinc"
DATA = ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/"
        "project4-hindi-decoder/data/persona_translation_v4/merged/")
XATTN_EVERY, XATTN_HEADS = 4, 8
MAX_NEW_TOKENS = 64
SEED = 42
DTYPE = torch.float32
WORK = "/kaggle/working"

assert torch.cuda.device_count() >= 2, "this notebook needs two GPUs (Kaggle: GPU T4 x2)"
ENC_DEVICE, DEC_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")
HF_TOKEN = HF_TOKEN.strip() or None
assert HF_TOKEN, "Gemma 3 is gated: paste a Hugging Face token in Cell 0"
login(HF_TOKEN)
api = HfApi()
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (ENCODER, DECODER)}
REV.update({r: api.dataset_info(r).sha for r in (TOP_REPO, PHINC_REPO)})
print({"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
       "gpus": [torch.cuda.get_device_name(i) for i in range(2)], "revisions": REV})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 81.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.9 MB/s eta 0:00:00:00:0100:01
{'torch': '2.11.0+cu128', 'transformers': '5.17.0', 'peft': '0.21.2', 'gpus': ['Tesla T4', 'Tesla T4'], 'revisions': {'answerdotai/ModernBERT-base': '8949b909ec900327062f0ebf497f51aef5e6f0c8', 'google/gemma-3-4b-it': '093f9f388b31de276ce2de164bdc2081324b9767', 'rvv-karma/English-Hinglish-TOP': '5ab10562888b4dd9735e434496f7e6112859329d', 'veezbo/phinc': 'a50e9b699599284036f934b3f5ccc42baa6724b7'}}


## Cell 2 — Persona data (v4 merged)

Encoder input `"<persona name>: <English>"`; target the reference Hinglish line.

In [4]:
# Cell 2 — v4 merged splits

def fetch_jsonl(name):
    return [json.loads(l) for l in urllib.request.urlopen(DATA + name).read().decode("utf-8").splitlines() if l]


MANIFEST = json.load(urllib.request.urlopen(DATA + "manifest.json"))
NAME = {p: v["name"] for p, v in MANIFEST["personas"].items()}
SPLITS = {s: [{**r, "persona": r["persona_id"], "enc_input": f"{NAME[r['persona_id']]}: {r['source']}"}
              for r in fetch_jsonl(f"{s}.jsonl")]
          for s in ("train", "validation", "test", "ood", "minimal_pairs")}
TRAIN, VAL = SPLITS["train"], SPLITS["validation"]
print({s: len(rs) for s, rs in SPLITS.items()})

{'train': 768, 'validation': 75, 'test': 75, 'ood': 114, 'minimal_pairs': 72}


## Cell 3 — Public corpora: download, clean, guard against leakage, sample

- **Hinglish-TOP:** training file; keep pairs with 3–30 words on both sides and no Devanagari.
- **PHINC:** remove @handles, links and #hashtags, then the same length and script rules, and an English/Hinglish length ratio between 0.5 and 2.
- Drop pairs whose English resembles any v4 evaluation sentence (word overlap ≥ 0.6); de-duplicate on the English.
- Hold out 200 pairs from each corpus; the training set is all Hinglish-TOP human pairs, cleaned PHINC (up to 12,000), and Hinglish-TOP synthetic pairs up to 24,000 in total.

In [5]:
# Cell 3 — Corpora

def download(repo, path, name):
    url = f"https://huggingface.co/datasets/{repo}/resolve/{REV[repo]}/{path}"
    local = f"/tmp/{name}"
    if not os.path.exists(local):
        urllib.request.urlretrieve(url, local)
    return local


top = pd.read_parquet(download(TOP_REPO, "data/train-00000-of-00001.parquet", "top_train.parquet"))
phinc = pd.read_csv(download(PHINC_REPO, "hinglish_english_parallel_corpus.csv", "phinc.csv"))
DEVA = re.compile(r"[ऀ-ॿ]")


def clean_tweet(t):
    t = re.sub(r"https?://\S+|www\.\S+", " ", str(t))
    t = re.sub(r"[@#]\w+", " ", t)
    return re.sub(r"\s+", " ", t).strip(" .…:-")


def usable(en, hi):
    ne, nh = len(en.split()), len(hi.split())
    return 3 <= ne <= 30 and 3 <= nh <= 30 and not DEVA.search(hi) and 0.5 <= nh / ne <= 2.0


pairs = {"top_human": [], "top_synthetic": [], "phinc": []}
for en, hi, gen in zip(top["en"], top["hi_en"], top["generated_by"]):
    en, hi = str(en).strip(), str(hi).strip()
    if usable(en, hi):
        pairs["top_human" if gen == "human" else "top_synthetic"].append((en, hi))
for hi, en in zip(phinc["Sentence"], phinc["English_Translation"]):
    en, hi = clean_tweet(en), clean_tweet(hi)
    if usable(en, hi):
        pairs["phinc"].append((en, hi))

words = lambda t: set(re.findall(r"[a-z0-9']+", t.lower()))
EVAL_WORDS = [words(r["source"]) for s in ("validation", "test", "ood", "minimal_pairs") for r in SPLITS[s]]


def leaks(en):
    w = words(en)
    return any(len(w & e) / len(w | e) >= 0.6 for e in EVAL_WORDS if w | e)


rng = random.Random(SEED)
for k in pairs:
    rng.shuffle(pairs[k])
seen, dropped = set(), Counter()


def take(items, n):
    """The first n usable pairs of a shuffled list: not seen before, not resembling a v4 evaluation sentence."""
    out = []
    for en, hi in items:
        if len(out) >= n:
            break
        key = en.lower()
        if key in seen:
            dropped["duplicate"] += 1
        elif leaks(en):
            dropped["resembles a v4 evaluation sentence"] += 1
        else:
            seen.add(key)
            out.append((en, hi))
    return out


CORPUS_TOTAL, PHINC_MAX = 24000, 12000
HELD = [("top", *p) for p in take(pairs["top_human"], 200)] + [("phinc", *p) for p in take(pairs["phinc"], 200)]
train_pairs = [("top", *p) for p in take(pairs["top_human"][200:], len(pairs["top_human"]))]
train_pairs += [("phinc", *p) for p in take(pairs["phinc"][200:], PHINC_MAX)]
train_pairs += [("top", *p) for p in take(pairs["top_synthetic"], max(0, CORPUS_TOTAL - len(train_pairs)))]
rng.shuffle(train_pairs)
print({k: len(v) for k, v in pairs.items()}, "usable before sampling | dropped while sampling", dict(dropped))


def corpus_record(src, en, hi):
    return {"corpus": src, "source": en, "target": hi, "enc_input": f"Plain: {en}", "references": [hi]}


CORPUS_TRAIN = [corpus_record(*p) for p in train_pairs]
CORPUS_HELD = [corpus_record(*p) for p in HELD]
print(f"stage 1 training pairs {len(CORPUS_TRAIN)} {dict(Counter(r['corpus'] for r in CORPUS_TRAIN))} | "
      f"held out {len(CORPUS_HELD)}")
for r in CORPUS_TRAIN[:4]:
    print(f"  {r['enc_input']}  ->  {r['target']}")

{'top_human': 6330, 'top_synthetic': 166215, 'phinc': 12855} usable before sampling | dropped while sampling {'duplicate': 746}
stage 1 training pairs 24000 {'phinc': 12000, 'top': 12000} | held out 400
  Plain: what an abuse  ->  kya gaali di hai
  Plain: I need to get to Alice's party by 8 pm, what time should I leave home?  ->  Mujhe Alice ki party me 8 pm tak pohanchne ki zaroorat haitoh mujhe ghar se kabb nikal na chahiye?
  Plain: Barha came to challenge us, by the way break your tv again on 18th and put it on youtube, be ready for  ->  Barha aaye the humse panga lene . By the way 18 th ko phir se tv phorh ke video utube par dal dena . ke liye tayyar raho
  Plain: Play Jay Z's 444  ->  Jay Z ke 444 ko bajao


## Cell 4 — Models (as `7-simple-bridge/`)

In [6]:
# Cell 4 — ModernBERT on GPU 0; Gemma 3 4B IT (4-bit) on GPU 1

from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

used = {d: torch.cuda.memory_allocated(d) / 2**30 for d in (ENC_DEVICE, DEC_DEVICE)}
if max(used.values()) > 0.5:
    raise RuntimeError("GPU memory already in use: restart the kernel and run all cells once, from the top.")

enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
enc_cfg = AutoConfig.from_pretrained(ENCODER, revision=REV[ENCODER])
if hasattr(enc_cfg, "reference_compile"):
    enc_cfg.reference_compile = False
encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], config=enc_cfg, dtype=DTYPE,
                                    attn_implementation="sdpa").to(ENC_DEVICE)
dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
decoder = Gemma3ForConditionalGeneration.from_pretrained(DECODER, revision=REV[DECODER], dtype=DTYPE, token=HF_TOKEN,
                                                         quantization_config=quant, device_map={"": DEC_DEVICE.index},
                                                         attn_implementation="eager")
START = dec_tok.bos_token_id
EOS = dec_tok.convert_tokens_to_ids("<end_of_turn>")
eos = decoder.generation_config.eos_token_id
STOPS = sorted(set((eos if isinstance(eos, list) else [eos]) + [EOS]))
PAD = dec_tok.pad_token_id
D_ENC, D_DEC = encoder.config.hidden_size, decoder.config.text_config.hidden_size
LAYERS = next(m for n, m in decoder.named_modules() if n.endswith("language_model.layers"))
N_LAYERS = len(LAYERS)
print(f"encoder width {D_ENC} | decoder width {D_DEC}, {N_LAYERS} layers | start {START}, end {EOS}")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

encoder width 768 | decoder width 2560, 34 layers | start 2, end 106


## Cell 5 — Bridge, cross-attention, LoRA, and the forward pass (as `7-simple-bridge/`)

In [7]:
# Cell 5 — Bridge, gated cross-attention, hooks, LoRA, loss and translation

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


class Bridge(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.proj = nn.Linear(d_in, d_out, bias=False)
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.proj(h))


class GatedCrossAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.norm = nn.RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True, bias=False)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, h, memory, pad):
        out, _ = self.attn(self.norm(h), memory, memory, key_padding_mask=pad, need_weights=False)
        return h + torch.tanh(self.gate) * out


class Memory:
    memory, pad = None, None


MEM = Memory()
XATTN_LAYERS = list(range(XATTN_EVERY - 1, N_LAYERS, XATTN_EVERY))
torch.manual_seed(SEED)
bridge = Bridge(D_ENC, D_DEC).to(ENC_DEVICE)
xattn = nn.ModuleDict({str(i): GatedCrossAttention(D_DEC, XATTN_HEADS) for i in XATTN_LAYERS}).to(DEC_DEVICE)


def make_hook(block):
    def hook(module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        h = block(h, MEM.memory, MEM.pad)
        return (h, *output[1:]) if isinstance(output, tuple) else h
    return hook


HOOKS = [LAYERS[i].register_forward_hook(make_hook(xattn[str(i)])) for i in XATTN_LAYERS]
decoder = prepare_model_for_kbit_training(decoder, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
dec_model = get_peft_model(decoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=rf".*language_model.*\.({'|'.join(PROJ)})"))
encoder = get_peft_model(encoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["Wqkv", "Wo", "Wi"]))
GROUPS = {"encoder LoRA": [p for n, p in encoder.named_parameters() if "lora_" in n],
          "bridge": list(bridge.parameters()),
          "cross-attention": [p for n, p in xattn.named_parameters() if not n.endswith("gate")],
          "gates": [b.gate for b in xattn.values()],
          "decoder LoRA": [p for n, p in dec_model.named_parameters() if "lora_" in n]}
for params in GROUPS.values():
    for p in params:
        p.requires_grad_(True)
LRS = {"encoder LoRA": 1e-4, "bridge": 3e-4, "cross-attention": 3e-4, "gates": 1e-2, "decoder LoRA": 5e-5}


def encode(texts):
    t = enc_tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(ENC_DEVICE)
    h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
    MEM.memory, MEM.pad = bridge(h).to(DEC_DEVICE), (t["attention_mask"] == 0).to(DEC_DEVICE)


def loss_on(records):
    encode([r["enc_input"] for r in records])
    seqs = [[START] + dec_tok(r["target"], add_special_tokens=False)["input_ids"][:96] + [EOS] for r in records]
    T = max(map(len, seqs))
    ids = torch.full((len(seqs), T), PAD)
    labels = torch.full((len(seqs), T), -100)
    mask = torch.zeros((len(seqs), T), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)], labels[i, 1:len(s)], mask[i, :len(s)] = torch.tensor(s), torch.tensor(s[1:]), 1
    return dec_model(input_ids=ids.to(DEC_DEVICE), attention_mask=mask.to(DEC_DEVICE), labels=labels.to(DEC_DEVICE),
                     token_type_ids=torch.zeros_like(ids).to(DEC_DEVICE)).loss


@torch.no_grad()
def translate(r):
    dec_model.eval()
    encoder.eval()
    encode([r["enc_input"]])
    start = torch.tensor([[START]], device=DEC_DEVICE)
    out = dec_model.generate(input_ids=start, attention_mask=torch.ones_like(start), do_sample=False, num_beams=1,
                             max_new_tokens=MAX_NEW_TOKENS, eos_token_id=STOPS, pad_token_id=PAD,
                             temperature=None, top_p=None, top_k=None)
    new = out[0, 1:].tolist()
    return {"output": dec_tok.decode(new, skip_special_tokens=True).strip(), "stopped": bool(new) and new[-1] in STOPS}


@torch.no_grad()
def mean_loss(records, batch=8):
    dec_model.eval()
    encoder.eval()
    total = 0.0
    for i in range(0, len(records), batch):
        chunk = records[i:i + batch]
        total += loss_on(chunk).item() * len(chunk)
    return round(total / len(records), 4)


def make_optimizer(steps):
    opt = torch.optim.AdamW([{"params": GROUPS[g], "lr": lr} for g, lr in LRS.items()], weight_decay=0.0)
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / steps))


def snapshot():
    return [p.detach().to("cpu", copy=True) for params in GROUPS.values() for p in params]


def restore(state):
    for p, s in zip([p for params in GROUPS.values() for p in params], state):
        p.data.copy_(s.to(p.device))


def gates():
    return [round(math.tanh(b.gate.item()), 3) for b in xattn.values()]


def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


def evaluate(records):
    rows = []
    for r in records:
        out = translate(r)
        rows.append({**{k: r.get(k) for k in ("id", "persona", "corpus", "source", "enc_input", "references")}, **out,
                     "chrf": max(chrf(out["output"], ref) for ref in r["references"])})
    score = {"n": len(rows), "chrf": round(statistics.mean(x["chrf"] for x in rows), 1),
             "stopped": round(sum(x["stopped"] for x in rows) / len(rows), 3),
             "distinct_outputs": len({x["output"] for x in rows})}
    return rows, score


def show(rows, n, seed=7):
    for x in random.Random(seed).sample(rows, min(n, len(rows))):
        print(f"  in : {x['enc_input']}\n  out: {x['output']}" + ("" if x["stopped"] else "  [did not stop]") +
              f"\n  ref: {x['references'][0]}\n")


print({k: f"{sum(p.numel() for p in v) / 1e6:.2f} M" for k, v in GROUPS.items()})

{'encoder LoRA': '3.38 M', 'bridge': '1.97 M', 'cross-attention': '209.74 M', 'gates': '0.00 M', 'decoder LoRA': '29.80 M'}


## Cell 6 — Stage 1: translation pre-training on the public corpora

One pass over ≈ 24,000 pairs, batches of 8, linear decay. Every 500 steps: held-out loss (400 pairs) and the gate values. After the pass: chrF, stopping and distinct outputs on 120 held-out pairs (60 from each corpus), with examples. The stage-1 weights are saved to `/kaggle/working/stage1.pt`.

In [8]:
# Cell 6 — Stage 1

S1_BATCH, S1_EVAL_EVERY = 8, 500
s1_steps = -(-len(CORPUS_TRAIN) // S1_BATCH)
opt, sched = make_optimizer(s1_steps)
S1_LOG = [{"step": 0, "held_out_loss": mean_loss(CORPUS_HELD)}]
print(S1_LOG[-1])
torch.cuda.reset_peak_memory_stats(DEC_DEVICE)
t0, run = time.perf_counter(), []
for step in range(1, s1_steps + 1):
    dec_model.train()
    encoder.train()
    loss = loss_on(CORPUS_TRAIN[(step - 1) * S1_BATCH: step * S1_BATCH])
    loss.backward()
    opt.step()
    sched.step()
    opt.zero_grad()
    run.append(loss.item())
    if step % S1_EVAL_EVERY == 0 or step == s1_steps:
        S1_LOG.append({"step": step, "train_loss": round(sum(run) / len(run), 4),
                       "held_out_loss": mean_loss(CORPUS_HELD), "gates": gates()})
        run = []
        print(S1_LOG[-1], f"| {time.perf_counter() - t0:.0f}s | peak GPU 1 "
              f"{torch.cuda.max_memory_allocated(DEC_DEVICE) / 2**30:.1f} GB")

torch.save(snapshot(), f"{WORK}/stage1.pt")
S1_PROBE = [r for r in CORPUS_HELD if r["corpus"] == "top"][:60] + [r for r in CORPUS_HELD if r["corpus"] == "phinc"][:60]
S1_ROWS, S1_SCORE = evaluate(S1_PROBE)
print("\nstage 1, held-out corpus pairs:", S1_SCORE)
show(S1_ROWS, 8)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


{'step': 0, 'held_out_loss': 9.4293}
{'step': 500, 'train_loss': 4.343, 'held_out_loss': 3.828, 'gates': [-0.023, 0.061, 0.096, -0.084, -0.087, 0.097, 0.16, 0.378]} | 1272s | peak GPU 1 9.6 GB
{'step': 1000, 'train_loss': 3.7819, 'held_out_loss': 3.5443, 'gates': [-0.02, 0.038, 0.12, -0.121, -0.162, 0.124, 0.253, 0.448]} | 2537s | peak GPU 1 9.6 GB
{'step': 1500, 'train_loss': 3.6043, 'held_out_loss': 3.3514, 'gates': [-0.013, 0.062, 0.139, -0.183, -0.32, 0.137, 0.3, 0.486]} | 3821s | peak GPU 1 9.6 GB
{'step': 2000, 'train_loss': 3.45, 'held_out_loss': 3.2389, 'gates': [-0.039, 0.053, 0.155, -0.233, -0.402, 0.121, 0.361, 0.553]} | 5094s | peak GPU 1 9.6 GB
{'step': 2500, 'train_loss': 3.3646, 'held_out_loss': 3.1304, 'gates': [-0.027, 0.056, 0.181, -0.264, -0.476, 0.155, 0.418, 0.582]} | 6380s | peak GPU 1 9.6 GB
{'step': 3000, 'train_loss': 3.283, 'held_out_loss': 3.0824, 'gates': [-0.019, 0.063, 0.188, -0.28, -0.491, 0.164, 0.436, 0.593]} | 7686s | peak GPU 1 9.6 GB

stage 1, held-o

## Cell 7 — Stage 2: persona fine-tuning on v4 (continuing from stage 1)

Same as `7-simple-bridge/`: micro-batches of 4 with 2-step accumulation, up to 6 epochs, early stopping after 2 epochs without a lower v4 validation loss, best epoch kept. Each epoch prints three sample translations.

In [9]:
# Cell 7 — Stage 2

MAX_EPOCHS, PATIENCE, MICRO, ACCUM = 6, 2, 4, 2
opt, sched = make_optimizer(MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM)))
SAMPLES = [VAL[0], VAL[4], VAL[8]]
S2_LOG = [{"epoch": 0, "validation_loss": mean_loss(VAL)}]
print(S2_LOG[-1])
for r in SAMPLES:
    print(f"    {r['enc_input'][:70]:70s} -> {translate(r)['output']}")
best = (S2_LOG[0]["validation_loss"], 0, snapshot())
since_best, rng, t0 = 0, random.Random(SEED), time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    dec_model.train()
    encoder.train()
    order = rng.sample(TRAIN, len(TRAIN))
    batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
    run = []
    for n, b in enumerate(batches, 1):
        loss = loss_on(b)
        (loss / ACCUM).backward()
        run.append(loss.item())
        if n % ACCUM == 0 or n == len(batches):
            opt.step()
            sched.step()
            opt.zero_grad()
    S2_LOG.append({"epoch": epoch, "train_loss": round(sum(run) / len(run), 4), "validation_loss": mean_loss(VAL),
                   "gates": gates()})
    print(S2_LOG[-1], f"| {time.perf_counter() - t0:.0f}s")
    for r in SAMPLES:
        print(f"    {r['enc_input'][:70]:70s} -> {translate(r)['output']}")
    if S2_LOG[-1]["validation_loss"] < best[0]:
        best, since_best = (S2_LOG[-1]["validation_loss"], epoch, snapshot()), 0
    else:
        since_best += 1
    if since_best >= PATIENCE:
        print(f"early stop: no lower validation loss for {PATIENCE} epochs")
        break
restore(best[2])
BEST_EPOCH = best[1]
print(f"kept epoch {BEST_EPOCH} (validation loss {best[0]})")

{'epoch': 0, 'validation_loss': 4.0575}
    Don: We should celebrate the progress without ignoring the remaining w -> Mujhe ye bhi yaad dilaayen ki ye sabhi sabse zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zyada zy
    Anthony Gonsalves: I will support the decision once everyone understan -> Mujhe ye bhi pata chalega ki, ye sabhi sabse pehle se hi ho chuke hai.
    Jai (Sholay): Don't confuse being busy with making progress.           -> "Ye sab kuch chhod do, humko kuch achha karna hai" - Modi
{'epoch': 1, 'train_loss': 3.1275, 'validation_loss': 3.4875, 'gates': [-0.028, 0.061, 0.179, -0.183, -0.511, 0.114, 0.417, 0.615]} | 271s
    Don: We should celebrate the progress without ignoring the remaining w -> Pehle sabki sunna chahiye. Pehle decision ko bhi support chahiye.
    Anthony Gonsalves: I will support the decision once everyone understan -> Pehle sabko clear karna, phir decision.
    

## Cell 8 — Translate v4 validation, test and out-of-domain; scores and examples

chrF against the best reference, stopping, distinct outputs, and word-for-word copies of training targets. For comparison, `7-simple-bridge/` (no pre-training) reached validation chrF 12.0 with 5 distinct outputs of 75.

In [10]:
# Cell 8 — v4 evaluation

train_targets = {r["target"].strip() for r in TRAIN}
RESULTS, SCORES = {}, {}
for split in ("validation", "test", "ood"):
    RESULTS[split], SCORES[split] = evaluate(SPLITS[split])
    SCORES[split]["copies_of_training_targets"] = sum(x["output"] in train_targets for x in RESULTS[split])
    print(split, SCORES[split])
print("\nexamples (validation, test, out-of-domain):")
show(RESULTS["validation"] + RESULTS["test"] + RESULTS["ood"], 12)

validation {'n': 75, 'chrf': 18.4, 'stopped': 0.96, 'distinct_outputs': 71, 'copies_of_training_targets': 0}
test {'n': 75, 'chrf': 17.8, 'stopped': 1.0, 'distinct_outputs': 73, 'copies_of_training_targets': 0}
ood {'n': 114, 'chrf': 18.4, 'stopped': 1.0, 'distinct_outputs': 110, 'copies_of_training_targets': 0}

examples (validation, test, out-of-domain):
  in : Don: Battery died, call you from home.
  out: Phone utha kar message karo.
  ref: Battery khatam. Ghar se call karunga.

  in : Jai (Sholay): I can accept a delay if you explain the reason and the new timeline.
  out: Pehle sabhi options check karo. Final decision phir se le lo.
  ref: Deri theek hai, agar tum wajah aur nayi samay seema batao.

  in : Anthony Gonsalves: Lost my earphones on the bus.
  out: Blue folder ka track lost, green folder ka track clear!
  ref: Earphones bus mein kho gaye, ab bus gaane sunegi!

  in : Don: The customer is confused because we gave them conflicting answers.
  out: Pehle kisi ne bola tha k

## Cell 9 — Save, settings, records

Both LoRAs and the bridge + cross-attention are saved under `/kaggle/working/` (download them if the result is worth keeping).

In [11]:
# Cell 9 — Save the trained parts; settings and records

encoder.save_pretrained(f"{WORK}/encoder-lora")
dec_model.save_pretrained(f"{WORK}/decoder-lora")
torch.save({"bridge": bridge.state_dict(), "xattn": xattn.state_dict(), "xattn_layers": XATTN_LAYERS}, f"{WORK}/bridge_xattn.pt")
print(json.dumps({"revisions": REV, "corpus": {"train": dict(Counter(r["corpus"] for r in CORPUS_TRAIN)),
                                               "held_out": len(CORPUS_HELD), "dropped": dict(dropped)},
                  "lrs": LRS, "stage1": {"batch": S1_BATCH, "log": S1_LOG, "score": S1_SCORE},
                  "stage2": {"log": S2_LOG, "kept_epoch": BEST_EPOCH}, "scores": SCORES}, indent=1))
print("\n----- RECORDS (JSON lines) -----")
for x in S1_ROWS:
    print(json.dumps({"split": "corpus_held_out", **x}, ensure_ascii=False))
for split, rows in RESULTS.items():
    for x in rows:
        print(json.dumps({"split": split, **x}, ensure_ascii=False))
print("----- END RECORDS -----")

{
 "revisions": {
  "answerdotai/ModernBERT-base": "8949b909ec900327062f0ebf497f51aef5e6f0c8",
  "google/gemma-3-4b-it": "093f9f388b31de276ce2de164bdc2081324b9767",
  "rvv-karma/English-Hinglish-TOP": "5ab10562888b4dd9735e434496f7e6112859329d",
  "veezbo/phinc": "a50e9b699599284036f934b3f5ccc42baa6724b7"
 },
 "corpus": {
  "train": {
   "phinc": 12000,
   "top": 12000
  },
  "held_out": 400,
  "dropped": {
   "duplicate": 746
  }
 },
 "lrs": {
  "encoder LoRA": 0.0001,
  "bridge": 0.0003,
  "cross-attention": 0.0003,
  "gates": 0.01,
  "decoder LoRA": 5e-05
 },
 "stage1": {
  "batch": 8,
  "log": [
   {
    "step": 0,
    "held_out_loss": 9.4293
   },
   {
    "step": 500,
    "train_loss": 4.343,
    "held_out_loss": 3.828,
    "gates": [
     -0.023,
     0.061,
     0.096,
     -0.084,
     -0.087,
     0.097,
     0.16,
     0.378
    ]
   },
   {
    "step": 1000,
    "train_loss": 3.7819,
    "held_out_loss": 3.5443,
    "gates": [
     -0.02,
     0.038,
     0.12,
     -0.121,
